# 🚀 [Master Guide] 2D 열기계 해석 기반 패키지 최적 설계 파이프라인

> **이 문서는 새로운 LLM이나 AI 에이전트가 본 프로젝트에 투입되었을 때, 이전 맥락(Context) 없이도 프로젝트의 물리적 배경, 데이터 구조, 6단계 최적화 파이프라인을 완벽히 이해하고 즉시 코드를 작성할 수 있도록 팩트 체크와 제약 조건(Constraints)을 명문화한 마스터 가이드입니다.**

---

## 📌 1. 프로젝트 사전 정보 (Project Context & Meta-Data)

### 1.1. 도메인 및 물리적 배경
* **분야:** 반도체 패키징 / 복합재 구조의 2D 열기계 유한요소해석(Thermo-mechanical FEA)
* **현상:** 열팽창계수(CTE) 불일치로 인한 휨(Warpage) 및 층간 박리(Delamination), 다이 크랙(Die Crack) 발생.
* **조건:** 300초에 걸친 극렬한 온도 사이클링(가열-유지-냉각 3 Steps).

### 1.2. 데이터셋 구조 (X와 Y)
* **입력 변수 (X):** `P1 ~ P6` (6개). 각 층의 기하학적 두께(Thickness)를 의미. 범위는 대략 0.1 ~ 1.0.
* **출력 변수 (Y):** 300초 동안 기록된 16개의 시계열(Time-series) 응력/변형 데이터.
* **핵심 Y 변수의 물리적 의미 (Named Selection 위치 기준):**
  1. `WarpMax`: 패키지 전체의 최대 열변형량 (최소화 메인 타겟)
  2. `T_Tip_Peel`: 계면 끝단의 수직 응력. **박리(Delamination)**의 직접적 원인 (최소화 메인 타겟)
  3. `T_Tip_Shear`: 계면 엇갈림 응력. **계면 피로(Fatigue)** 유발
  4. `T_Tip_SEQV (Von Mises)`: 끝단 등가 응력. **소성 변형(Plasticity)** 유발
  5. `T_Avg_Peel` / `T_Avg_Shear`: 접합면 전체 평균 응력. **중앙부 거품(Void)** 유발
  6. `Die_SX` (Die Bending Stress): 실리콘 칩 본체의 휨 응력. **다이 크랙(Die Crack)** 유발
  7. `Die_Corner_Stress`: 칩 모서리 응력 집중도

### 1.3. 데이터 결측치 (Missing Data / Infeasibility)
* 계획된 1200개의 DP(Design Point) 중 약 **29%는 시뮬레이션 중 메쉬 꼬임이나 심각한 박리로 인해 해석이 터져버림 (결측치 발생).**
* 살아남은 71%의 데이터만 시계열 CSV 파일(`ML_DATA_Extract_Row_N.csv`)로 존재함.

---

## 📌 2. 6단계 최적화 파이프라인 (Action Plan)

본 프로젝트는 단순히 대리 모델을 만드는 것을 넘어, 역설계(Inverse Design)와 유전 알고리즘(GA)을 결합하여 최적의 P1~P6를 도출하는 End-to-End 프레임워크입니다. **코드 작성 시 아래의 각 단계별 지침과 [🚨 경고] 사항을 반드시 준수해야 합니다.**

### Step 1: 대리 모델(Surrogate)을 통한 데이터 증강 (Data Augmentation)
* **목표:** 800여 개의 생존 데이터 한계를 극복하기 위해 가상 데이터를 생성.
* **방법:** 생존한 데이터의 '절댓값 Max Peak' 지표들을 추출하여 머신러닝(XGBoost 또는 GPR) 학습. 이후 난수 생성기(Monte Carlo)로 10만 개의 가상 P1~P6 조합을 만들고 Y값들을 예측함.

### Step 2: 은닉 제약조건 분류기(Gatekeeper)를 통한 필터링
* **목표:** 물리적으로 파괴되는(해석이 터지는) 치수 조합을 사전에 걸러냄.
* **방법:** 원본 1200개 DP의 CSV 파일 존재 여부를 1(Safe)과 0(Fail)으로 라벨링하여 Random Forest 분류기 학습. Step 1에서 만든 10만 개의 가상 조합을 이 분류기에 통과시켜, **생존 확률 95% 이상인 안전한 가상 데이터**만 필터링함.

### Step 3: 파레토 프론티어 (Pareto Frontier) 타겟 곡선 추출
* **목표:** 역설계 AI에 입력할 '물리적으로 도달 가능하면서도 완벽에 가까운 타겟 시계열 텐서' 생성.
* **[🚨 경고 - 치명적 오류 주의]:** * 300초 '평균값'이 아닌 **반드시 시계열 전체의 '절댓값 최대 피크(Max Peak)'**를 기준으로 우수 데이터를 선별할 것. [Image of thermal cycling stress hysteresis loop]
  * Von Mises 등가 응력이 아닌 **`WarpMax`와 `T_Tip_Peel` 단 2가지만을 기준**으로 파레토 최적 DP(상위 5~10%)를 선별할 것.
* **추출 및 스케일링 방법:** 선정된 파레토 DP의 300초 Raw 시계열 곡선을 8~9개 주요 채널 모두 통째로 가져옴. 이후 모든 채널에 **동일한 스칼라(예: x0.9)**를 곱하여 진폭만 10% 낮춘 '유토피아 타겟 다채널 텐서'를 생성. (물리적 위상차 보존)

### Step 4: 딥러닝 기반 역설계 (Inverse Design) 초안 출력
* **목표:** 타겟 곡선을 입력하면 이를 구현할 수 있는 최적의 P1~P6 초안(Draft) 도출.
* **방법:** 1D-CNN 역방향 모델 또는 오토인코더(Autoencoder) 잠재 매핑 모델을 사용. Step 3에서 만든 '유토피아 타겟 텐서'를 입력하면 AI가 1차 P1~P6 설계안을 한 번의 연산으로 출력함. (8채널 동시 입력 구조로 전체 응력 밸런스 학습)

### Step 5: 머신러닝 미세 튜닝 (Fine-tuning via GA & Penalty Limits)
* **목표:** 도출된 초안을 바탕으로 유전 알고리즘(NSGA-II)을 돌려 최종 최적화 및 물리적 한계치(Limit) 강제 적용.
* **방법:** 1. Step 4의 초안 치수 기준 $\pm 10\%$ 내외로 좁은 탐색 바운더리 설정.
  2. 목적 함수(Loss)는 `WarpMax`와 `T_Tip_Peel` 가중치 합산으로 최소화.
  3. **[🚨 필수 - Hard Constraints]:** 최적화 과정에서 나머지 응력들이 재료의 물리적 한계치(Limit)를 넘을 경우, Loss에 +999,999점의 페널티를 부여하여 즉시 도태시킴.
     * Limit 예시: `Die_SX` < 실리콘 파괴 인성, `T_Tip_Shear` < 계면 피로 한계 등. [Image of Pareto front in multi-objective optimization]

### Step 6: 디지털 트윈 (Digital Twin) 최종 시뮬레이션 검증
* **목표:** AI가 제시한 최종 P1~P6 조합을 실제 Ansys 2D 해석에 입력.
* **방법:** 도출된 시계열 곡선 결과를 베이스라인(초기 설계) 및 Step 3의 유토피아 타겟과 중첩 플롯(Overlay Plot)하여 휨 및 박리 응력의 저감률을 검증.

